# Column-wise char TF-IDF + Logistic Regression

各foldのtrainingだけで列別TF-IDFをfitし、単独列モデルと3列結合モデルを比較します。生成特徴量・OOF・スコアは`data/csv/`へ保存します。

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == 'notebooks' else WORKING_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from text_features import (
    compare_cv_results,
    cross_validate_single_text_column,
    cross_validate_text_columns,
    fit_full_text_model_predict,
    show_top_coefficients,
)
from validation import encode_binary_target, make_time_series_cv

## Configuration

In [ ]:
INPUT_DIR = PROJECT_ROOT / 'input'
FEATURE_DIR = PROJECT_ROOT / 'data' / 'csv'
TRAIN_PATH = INPUT_DIR / 'train.csv'
TEST_PATH = INPUT_DIR / 'test.csv'

TEXT_COLS = ['project_name', 'project_objective', 'project_summary', 'current_issues']
TARGET_COL = 'science_tech_decision'
PROJECT_COL = 'project_name'
YEAR_COL = 'project_start_year'
N_VALID_YEARS = 3
METRIC = 'roc_auc'

TFIDF_PARAMS = {
    'analyzer': 'char',
    'ngram_range': (2, 5),
    'min_df': 2,
    'max_features': 300_000,
    'sublinear_tf': True,
}
LOGREG_PARAMS = {
    'C': 1.0,
    'max_iter': 3000,
    'solver': 'liblinear',
    'dual': True,
    'random_state': 42,
}

## Load data and folds

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train[TARGET_COL] = encode_binary_target(train[TARGET_COL])  # 該当=1, 非該当=0
print('train:', train.shape, 'test:', test.shape)

folds, diagnostics = make_time_series_cv(
    df=train,
    year_col=YEAR_COL,
    project_col=PROJECT_COL,
    target_col=TARGET_COL,
    n_valid_years=N_VALID_YEARS,
)
display(diagnostics)

## Evaluate each text column

各実験の特徴量は`data/csv/<model_name>/`へ疎なlong形式のCSVとして保存されます。

In [ ]:
results = []
for text_col in TEXT_COLS:
    result = cross_validate_single_text_column(
        df=train,
        folds=folds,
        text_col=text_col,
        target_col=TARGET_COL,
        project_col=PROJECT_COL,
        year_col=YEAR_COL,
        model_name=f'{text_col}_char_tfidf',
        metric=METRIC,
        tfidf_params=TFIDF_PARAMS,
        logreg_params=LOGREG_PARAMS,
        feature_output_dir=FEATURE_DIR,
    )
    results.append(result)
    display(result['fold_scores'])

## Evaluate separately-vectorized columns with hstack

In [ ]:
combined_result = cross_validate_text_columns(
    df=train,
    folds=folds,
    text_cols=TEXT_COLS,
    target_col=TARGET_COL,
    project_col=PROJECT_COL,
    year_col=YEAR_COL,
    model_name='all_columns_char_tfidf',
    metric=METRIC,
    tfidf_params=TFIDF_PARAMS,
    logreg_params=LOGREG_PARAMS,
    feature_output_dir=FEATURE_DIR,
)
results.append(combined_result)
display(combined_result['fold_scores'])

## Compare experiments

In [ ]:
comparison = compare_cv_results(results)
display(comparison)
comparison.to_csv(FEATURE_DIR / 'experiment_comparison.csv')

## Fit all train rows and predict test

testはvectorizerのfitに使用しません。最終特徴量と予測も`data/csv/`へ保存されます。

In [ ]:
final_result = fit_full_text_model_predict(
    train_df=train,
    test_df=test,
    text_cols=TEXT_COLS,
    target_col=TARGET_COL,
    model_name='all_columns_char_tfidf_final',
    tfidf_params=TFIDF_PARAMS,
    logreg_params=LOGREG_PARAMS,
    feature_output_dir=FEATURE_DIR,
)
test_pred = final_result['test_pred']
display(test_pred.head())

## Top coefficients

In [ ]:
top_coefficients = show_top_coefficients(
    vectorizers=final_result['vectorizers'],
    model=final_result['model'],
    top_n=50,
)